Codigo df

In [2]:
%pip install nltk

  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.7 MB ? eta -:--:--
   ------------ --------------------------- 0.5/1.7 MB 659.5 kB/s eta 0:00:02
   ------------ --------------------------- 0.5/1.7 MB 659.5 kB/s eta 0:00:02
   ------------------ --------------------- 0.8/1.7 MB 666.9 kB/s eta 0:00:02
   ------------------ --------------------- 0.8/1.7 MB 666.9 kB/s eta 0:00:02
   ------------------------ --------------- 1.0/1.7 MB 639.3 kB/s eta 0:00:02
   ------------------------------ --------- 1.3/1.7 MB 714.4 kB/s eta 0:00:01
   ------------------------------ --------- 1.3/1.7 MB 7

In [3]:
import nltk

nltk.download('gutenberg')
nltk.download('punkt')  # Requerido para la tokenización de oraciones

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\andyj\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\gutenberg.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\andyj\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\andyj\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [4]:
from nltk.corpus import gutenberg

# Lista con los 18 identificadores de archivo
file_ids = gutenberg.fileids()
print("Archivos disponibles (18):")
for f in file_ids:
    print(f"- {f}")


Archivos disponibles (18):
- austen-emma.txt
- austen-persuasion.txt
- austen-sense.txt
- bible-kjv.txt
- blake-poems.txt
- bryant-stories.txt
- burgess-busterbrown.txt
- carroll-alice.txt
- chesterton-ball.txt
- chesterton-brown.txt
- chesterton-thursday.txt
- edgeworth-parents.txt
- melville-moby_dick.txt
- milton-paradise.txt
- shakespeare-caesar.txt
- shakespeare-hamlet.txt
- shakespeare-macbeth.txt
- whitman-leaves.txt


In [10]:
import time
import re
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import gutenberg

# 1. Cargar textos originales e identificar documentos y sus tamaños
file_ids = gutenberg.fileids()
print(f"Número total de documentos: {len(file_ids)}\n")

print(f"{'ID Documento':<30} | {'Caracteres':<12} | {'Palabras (raw)':<15}")
print("-" * 65)
for fid in file_ids:
    raw_len = len(gutenberg.raw(fid))
    words_len = len(gutenberg.words(fid))
    print(f"{fid:<30} | {raw_len:<12,} | {words_len:<15,}")

# Configuration de preprocesamiento consistente:
# - Conversión a minúsculas
# - Eliminación de puntuación y caracteres especiales (solo se conservan caracteres alfanuméricos)
def preprocess_text(text):
    text_lower = text.lower()
    # Extraer únicamente tokens alfanuméricos
    tokens = re.findall(r'\b\w+\b', text_lower)
    return tokens

# Procesar y almacenar los documentos en memoria para los experimentos
corpus_preprocessed = {
    fid: preprocess_text(gutenberg.raw(fid)) 
    for fid in file_ids
}

print("\nConfiguración de preprocesamiento completada.")

Número total de documentos: 18

ID Documento                   | Caracteres   | Palabras (raw) 
-----------------------------------------------------------------
austen-emma.txt                | 887,071      | 192,427        
austen-persuasion.txt          | 466,292      | 98,171         
austen-sense.txt               | 673,022      | 141,576        
bible-kjv.txt                  | 4,332,554    | 1,010,654      
blake-poems.txt                | 38,153       | 8,354          
bryant-stories.txt             | 249,439      | 55,563         
burgess-busterbrown.txt        | 84,663       | 18,963         
carroll-alice.txt              | 144,395      | 34,110         
chesterton-ball.txt            | 457,450      | 96,996         
chesterton-brown.txt           | 406,629      | 86,063         
chesterton-thursday.txt        | 320,525      | 69,213         
edgeworth-parents.txt          | 935,158      | 210,663        
melville-moby_dick.txt         | 1,242,990    | 260,819        
milton

Inicio para Matriz TF

In [13]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# 1. Reconstruir el texto procesado uniendo los tokens por espacio
corpus_texts = {
    fid: " ".join(tokens) 
    for fid, tokens in corpus_preprocessed.items()
}

# 2. Inicializar CountVectorizer con token_pattern consistente con el preprocesamiento
vectorizer = CountVectorizer(token_pattern=r"\b\w+\b")

# 3. Construir la matriz TF (Documentos x Términos)
tf_matrix_sparse = vectorizer.fit_transform(corpus_texts.values())

# 4. Crear DataFrame para la matriz Documento-Término
tf_df = pd.DataFrame(
    tf_matrix_sparse.toarray(),
    index=list(corpus_texts.keys()),
    columns=vectorizer.get_feature_names_out()
)

# 5. Matriz Término-Documento (formato clásico de Recuperación de Información)
tf_term_doc = tf_df.T
tf_term_doc.index.name = "Término"

print(f"Dimensiones Matriz Documento-Término: {tf_df.shape} (Documentos x Términos)")
print(f"Dimensiones Matriz Término-Documento: {tf_term_doc.shape} (Términos x Documentos)\n")

# Visualizar las primeras filas de la matriz Término-Documento
tf_term_doc.head(10)


Dimensiones Matriz Documento-Término: (18, 42099) (Documentos x Términos)
Dimensiones Matriz Término-Documento: (42099, 18) (Términos x Documentos)



,austen-emma.txt,austen-persuasion.txt,austen-sense.txt,bible-kjv.txt,blake-poems.txt,bryant-stories.txt,burgess-busterbrown.txt,carroll-alice.txt,chesterton-ball.txt,chesterton-brown.txt,chesterton-thursday.txt,edgeworth-parents.txt,melville-moby_dick.txt,milton-paradise.txt,shakespeare-caesar.txt,shakespeare-hamlet.txt,shakespeare-macbeth.txt,whitman-leaves.txt
Término,,,,,,,,,,,,,,,,,,
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
00,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
000,2,0,0,0,0,0,0,0,1,0,0,0,20,0,0,0,0,0
00021053,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
00081429,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
00482129,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
01,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
02,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0
1,0,3,2,2783,0,0,0,0,7,0,0,0,2,0,19,4,36,36


In [16]:
# Normalización por longitud de documento
tf_normalized_df = tf_df.div(tf_df.sum(axis=1), axis=0)
tf_normalized_df.head()

Término,0,00,000,00021053,00081429,00482129,01,02,1,10,...,zorobabel,zso,zuar,zumpt,zuph,zur,zuriel,zurishaddai,zuyder,zuzims
austen-emma.txt,0.0,0.0,0.000012,0.0,0.0,0.0,0.0,0.0,0.000000,0.000012,...,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
austen-persuasion.txt,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000036,0.000012,...,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
austen-sense.txt,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000017,0.000008,...,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
bible-kjv.txt,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.003259,0.002479,...,0.000004,0.0,0.000006,0.0,0.000004,0.000006,0.000001,0.000006,0.0,0.000001
blake-poems.txt,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,...,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000


In [17]:
# DF: Contar en cuántos documentos aparece cada palabra (DF > 0)
df_series = (tf_df > 0).sum(axis=0)

# Convertir a DataFrame para visualización
df_table = pd.DataFrame(df_series, columns=['DF'])
df_table.head()

,DF
Término,
0,1
00,1
000,3
00021053,1
00081429,1


In [ ]:
N = len(tf_df)  # Número total de documentos (18 en Gutenberg)
idf_series_raw = np.log(N / df_series)

In [22]:
# Multiplicación elemento por elemento
tfidf_df = tf_df.multiply(idf_series_raw, axis=1)

print(f"Dimensiones de la matriz TF-IDF: {tfidf_df.shape}")
tfidf_df.head()

Dimensiones de la matriz TF-IDF: (18, 42099)


Término,0,00,000,00021053,00081429,00482129,01,02,1,10,...,zorobabel,zso,zuar,zumpt,zuph,zur,zuriel,zurishaddai,zuyder,zuzims
austen-emma.txt,0.0,0.0,3.583519,0.0,0.0,0.0,0.0,0.0,0.000000,1.888923,...,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
austen-persuasion.txt,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,2.079442,0.944462,...,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
austen-sense.txt,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,1.386294,0.944462,...,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
bible-kjv.txt,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,1929.028603,1999.425226,...,8.671115,0.0,14.451859,0.0,8.671115,14.451859,2.890372,14.451859,0.0,2.890372
blake-poems.txt,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,...,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
